# 第 2 章：使用文本数据
此笔记本中使用的软件包：

In [1]:
from importlib.metadata import version
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.11.0
tiktoken version: 0.12.0


## 2.1 理解词嵌入

### 核心概念

**为什么需要嵌入？**
- 神经网络无法直接处理离散的文本数据
- 需要将文本转换为连续值向量（张量），才能进行数学运算

**什么是嵌入（Embedding）？**
- 将离散对象（单词、图像等）映射到连续向量空间的过程
- 本质：非数值数据 → 神经网络可处理的数值格式

### 不同数据类型的嵌入

| 数据类型 | 数据特征 | 嵌入模型 | 主要特征 |
|---------|---------|---------|---------|
| 文本 | 离散的、序列化的符号数据 | Word2Vec, GloVe, BERT, GPT | 语义关系、上下文理解 |
| 图像 | 二维像素网格，具有空间特征 | CNN（ResNet、VGG）、ViT | 形状、纹理、颜色等视觉特征 |
| 音频 | 一维时序信号 | CNN+频谱图、RNN、Transformer | 频率、音调、时序依赖 |
| 视频 | 时空序列数据 | 3D CNN、RNN+CNN、Video Transformer | 时空特征、动作捕捉 |

### 词嵌入的关键特点

**维度选择**
- 维度可变：从几维到上万维
- GPT-2 (125M参数)：768维
- GPT-3 (175B参数)：12,288维
- 高维优势：捕捉更细微的语义关系
- 高维代价：计算效率降低

**LLM中的嵌入方式**
- 不直接使用预训练模型（如Word2Vec）
- LLM自己生成并优化嵌入
  - 嵌入层是模型的一部分
  - 在训练过程中不断更新
  - 针对特定任务和数据优化

# 2.2文本分词
### 核心概念

**什么是分词？**
- 将输入文本拆分为单个 token 的过程
- Token 可以是：单个单词、特殊字符、标点符号
- 是创建 LLM 嵌入的必要预处理步骤

**为什么需要分词？**
- LLM 无法直接处理原始文本字符串
- 需要将文本转换为离散的 token 单元
- 每个 token 后续会被转换为向量表示

In [2]:
#自行准备对应的txt文本
with open("the-verdict.txt", "r",encoding="utf-8") as f:
    novel_text = f.read()
print("文本总长度为:", len(novel_text))
print(novel_text[:500])

文本总长度为: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it'


下一步的目标是把小说文本拆分成token单元，并将这些token转换为LLM训练所需的张量。

In [3]:
#使用py的正则表达式库进行简单的分词，仅考虑单词和标点符号
import re
text = "Hello, world! This is a simple tokenization example.";
#注意，中文无法如此分词
tokens = re.split(r'(\s)', text)
print(tokens)

['Hello,', ' ', 'world!', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'simple', ' ', 'tokenization', ' ', 'example.']


In [4]:
#将单词和标点符号进行拆分
tokens = re.split(r'(\W+)', text)
print(tokens)
#对于空白字符，删除空白字符可以得到更干净的分词结果，并减少计算的消耗
#但对于如果训练的模型对于空白字符敏感，那么空白字符的保留可能会有作用，如python代码输入文本中包含多个空格或者制表符。
#我们选择删除空白字符，仅保留单词和标点
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ', ', 'world', '! ', 'This', ' ', 'is', ' ', 'a', ' ', 'simple', ' ', 'tokenization', ' ', 'example', '.', '']
['Hello', ',', 'world', '!', 'This', 'is', 'a', 'simple', 'tokenization', 'example', '.']


现在把分词器运用到小说文本中

In [5]:
#把小说分成单词和标点
tokens = re.split(r'([,.:;?_!"()\']|--|\s)', novel_text)
tokens = [item.strip() for item in tokens if item.strip()]
print("分词后的单词数：", len(tokens))


分词后的单词数： 4690


## 2.3 把token转换成token id
把tokens变成token ids，方便后续把token ids变成嵌入向量
用前面小说文本分词得到的tokens构建一个词汇表，token按照字典序排序，并且只包含唯一的token

In [6]:
words = sorted(set(tokens))
print("词汇表大小：", len(words))
print("词汇表前20个token：", words[:20])
vocabs ={token:integer for integer, token in enumerate(words)}
for i,item in enumerate(vocabs.items()):
    print(item)
    if i>50:
        break

词汇表大小： 1130
词汇表前20个token： ['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be']
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)


上一步我们构建了一个词汇表，把每个token映射到一个唯一的整数id。现在我们可以使用这个词汇表把小说文本中的每个token转换成对应的token id。
下一步构建一个分词器，包含encode和decode，encode方法把文本转化成token，然后转换成token id，decode方法把token id转换成token，再转换成文本。

In [7]:
class XinTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {integer: token for token, integer in vocab.items()}
    def encode(self, xin_text):
        xin_tokens = re.split(r'([,.:;?_!"()\']|--|\s)', xin_text)
        xin_tokens = [item.strip() for item in xin_tokens if item.strip()]
        ids = [self.str_to_int[s] for s in xin_tokens]
        return ids
    def decode(self, ids):
        xin_text = " ".join([self.int_to_str[i] for i in ids])
        #去除多余的空格
        xin_text = re.sub(r'\s+([,.:;?_!"()\']|--)', r'\1 ', xin_text)
        return xin_text

使用自定义的分词器把小说文本转换成token id，并且把token id转换回文本，验证分词器的正确性。


In [8]:
tokenizer = XinTokenizerV1(vocabs)
token_ids = tokenizer.encode(novel_text)
print("token ids的长度：", len(token_ids))
print("前20个token ids：", token_ids[:20])
#把token ids转换回文本
decoded_text = tokenizer.decode(token_ids)
print("解码后的文本前500个字符：", decoded_text[:500])
# tokenizer.decode(tokenizer.encode(novel_text))


token ids的长度： 4690
前20个token ids： [53, 44, 149, 1003, 57, 38, 818, 115, 256, 486, 6, 1002, 115, 500, 435, 392, 6, 908, 585, 1077]
解码后的文本前500个字符： I HAD always thought Jack Gisburn rather a cheap genius--  though a good fellow enough--  so it was no great surprise to me to hear that,  in the height of his glory,  he had dropped his painting,  married a rich widow,  and established himself in a villa on the Riviera. (  Though I rather thought it would have been Rome or Florence. ) "  The height of his glory" --  that was what the women called it.  I can hear Mrs.  Gideon Thwing--  his last Chicago sitter--  deploring his unaccountable abdic


## 2.4处理不在词表中的token
不在词表中的token将无法被编码成token id，使用encode方法时会抛出KeyError异常。我们可以测试一下这个分词器对于不在词表中的token的处理。

In [9]:
# tokenizer = XinTokenizerV1(vocabs)
#
# text = "Hello, do you like tea. Is this-- a test?"
#
# tokenizer.encode(text)

修改词表，添加特殊的token来处理不在词表中的token，例如添加一个特殊的token "<UNK>"来表示未知的token，并且在encode方法中把不在词表中的token编码成"<UNK>"的token id。

In [10]:
all_tokens = sorted(list(set(tokens)))
all_tokens.extend(["<UNK>","<EOF>"])
vocabs = {token: integer for integer, token in enumerate(all_tokens)}
print("新的词汇表大小：", len(vocabs))

新的词汇表大小： 1132


In [11]:
#输出词表末尾来查看
for i, item in enumerate(list(vocabs.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<UNK>', 1130)
('<EOF>', 1131)


修改分词器的代码，让其可以处理未知的token，在encode方法中把不在词表中的token编码成"<UNK>"的token id。

In [12]:
class XinTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {integer: token for token, integer in vocab.items()}
    def encode(self, xin_text):
        xin_tokens = re.split(r'([,.:;?_!"()\']|--|\s)', xin_text)
        xin_tokens = [item.strip() for item in xin_tokens if item.strip()]
        #对于不在词表中的token，使用"<UNK>"的token id
        xin_tokens = [s if s in self.str_to_int else "<UNK>" for s in xin_tokens]

        ids = [self.str_to_int[s] for s in xin_tokens]
        return ids
    def decode(self, ids):
        xin_text = " ".join([self.int_to_str[i] for i in ids])
        #去除多余的空格
        xin_text = re.sub(r'\s+([,.:;?_!"()\']|--)', r'\1 ', xin_text)
        return xin_text

再次尝试之前报错的代码


In [13]:
tokenizer = XinTokenizerV2(vocabs)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <EOF> ".join((text1, text2))

print(text)

tokenizer.decode(tokenizer.encode(text))

Hello, do you like tea? <EOF> In the sunlit terraces of the palace.


'<UNK>,  do you like tea?  <EOF> In the sunlit terraces of the <UNK>. '

## 2.5字节对编码（BPE）
字节对编码（BPE）是一种基于统计的分词方法。
BPE的核心思想是通过迭代地合并最频繁出现的字符对来构建一个词汇表，从而实现对文本的有效分词和压缩。BPE在处理自然语言文本时，能够捕捉到常见的子词单元，使得模型能够更好地理解和生成文本。BPE的基本步骤如下：
1. 初始化：将文本中的每个字符视为一个独立的 token，构建一个初始的词汇表。
2. 统计频率：计算文本中所有相邻字符对的频率。
3. 合并：找到频率最高的字符对

可自己实现bpe算法，或者使用现成的库，如 `tiktoken` 库中的 BPE 实现。

In [14]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [15]:
#bpe可以处理不在词表中的token，得益于bpe的子词单元化特性，能够将未知的token分解成已知的子词单元，从而实现对未知token的编码和处理。
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.max_token_value)
text = "Hello, do you like tea? teeeaal?哈哈哈？《123》<|EOF|><|endoftext|>"
token_ids = tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print("Token IDs:", token_ids)
decoded_text = tokenizer.decode(token_ids)
print("Decoded Text:", decoded_text)


50256
Token IDs: [15496, 11, 466, 345, 588, 8887, 30, 573, 1453, 64, 282, 30, 161, 241, 230, 161, 241, 230, 161, 241, 230, 171, 120, 253, 5099, 232, 10163, 5099, 233, 27, 91, 4720, 37, 91, 29, 50256]
Decoded Text: Hello, do you like tea? teeeaal?哈哈哈？《123》<|EOF|><|endoftext|>


## 2.6使用滑动窗口进行数据采样
在先前的章节中，我们已经将文本数据转换成了 token id 的形式。接下来，我们需要将这些 token id 组织成适合训练语言模型的样本。

滑动窗口是一种常用的数据采样方法，特别适用于处理长文本数据。它通过在文本上定义一个固定大小的窗口，并以一定的步长滑动该窗口来生成训练样本。这种方法可以有效地利用文本中的上下文信息，同时避免了处理过长文本时的内存问题。

本节将会实现一个数据加载器，使用滑动窗口方法从 token id 序列中生成训练样本。每个样本将包含一个输入序列和一个目标序列，输入序列是当前窗口内的 token id，而目标序列是输入序列向右移动一位后的 token id。这种设置使得模型能够学习预测下一个 token，从而进行语言建模任务。

In [16]:
#首先使用之前构建的分词器把小说文本转换成token id，并且把token id转换回文本，验证分词器的正确性。
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    novel = f.read()
enc_text = tokenizer.encode(novel)
print("Token IDs的长度：", len(enc_text))

Token IDs的长度： 5145


接下来我们定义一个函数，使用滑动窗口方法从 token id 序列中生成训练样本。每个样本将包含一个输入序列和一个目标序列，输入序列是当前窗口内的 token id，而目标序列是输入序列向右移动一位后的 token id。

In [17]:
#移除前50个token id
enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


输出结果显示，输入序列 x 包含了当前窗口内的 token id，而目标序列 y 是输入序列向右移动一位后的 token id。这种设置使得模型能够学习预测下一个 token，从而进行语言建模任务。

In [18]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [19]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


前置知识已经足够，现在来构造一个数据加载器。


In [20]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


这个类的作用是将原始文本转换成重叠的输入-目标序列对，用于 GPT 模型的训练。

In [21]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    #各个参数的含义如下：
    # txt: 原始文本数据
    # tokenizer: 用于将文本转换为 token id 的分词器
    # max_length: 每个输入序列的最大长度
    # stride: 滑动窗口的步长，控制输入序列之间的重叠程度
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 首先将原始文本转换为 token id 序列
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # 使用滑动窗口方法生成输入-目标序列对
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    # 返回数据集总行数
    def __len__(self):
        return len(self.input_ids)
    # 根据索引返回对应的输入序列和目标序列
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

然后通过pytorch的dataLoader来加载数据集，设置合适的batch size和shuffle参数。

In [22]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

设置合适的参数来创建数据加载器，并查看生成的输入-目标序列对。
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">


In [23]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [24]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=2, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464],
        [1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899]])]
[tensor([[10899,  2138,   257,  7026],
        [15632,   438,  2016,   257]]), tensor([[ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922]])]


## 2.7构建词嵌入
前面的章节我们已经将文本数据转换成了 token id 的形式，并且使用滑动窗口方法生成了输入-目标序列对。接下来，我们需要将这些 token id 转换成适合训练语言模型的嵌入向量。

词嵌入（Word Embeddings）是将离散的 token id 转换为连续的向量表示的过程。这些向量能够捕捉到 token 之间的语义关系，使得模型能够更好地理解和生成文本。

在本节中，我们将实现一个简单的词嵌入层，将 token id 转换为嵌入向量。我们将使用 PyTorch 来实现这个词嵌入层，并将其集成到我们的数据加载器中，以便在训练语言模型时能够直接获取嵌入向量。

In [25]:
#假设我们有以下四个输入示例，输入ID分别为2、3、5和1
input_ids = torch.tensor([2, 3, 5, 1])
#为了简单起见，假设我们的词汇量很小，只有 6 个单词，并且我们想要创建大小为 3 的嵌入
vocab_size = 6
output_dim = 3

torch.manual_seed(225)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
#这将产生一个 6x3 权重矩阵：
print(embedding_layer.weight)


Parameter containing:
tensor([[-0.3881, -0.6688,  0.3641],
        [-2.4813, -2.2219, -0.8954],
        [-0.6317, -0.0696,  1.2939],
        [ 0.0413, -2.0817,  0.8616],
        [-0.6195, -1.2197, -0.0335],
        [-0.3613,  0.9926,  0.7971]], requires_grad=True)


有了这个嵌入层，我们可以将输入ID转换为嵌入向量。每个输入ID将被映射到一个对应的嵌入向量。

In [26]:
print(embedding_layer(input_ids))

tensor([[-0.6317, -0.0696,  1.2939],
        [ 0.0413, -2.0817,  0.8616],
        [-0.3613,  0.9926,  0.7971],
        [-2.4813, -2.2219, -0.8954]], grad_fn=<EmbeddingBackward0>)


# 2.8位置编码
位置编码（Positional Encoding）是 Transformer 模型中的一个重要组件，用于为输入序列中的每个 token 提供位置信息。由于 Transformer 模型是基于自注意力机制的，它本身并不具备处理序列顺序的能力，因此需要通过位置编码来引入位置信息，使得模型能够理解 token 之间的相对位置关系。

有两种位置编码方法：绝对位置编码和相对位置编码。绝对位置编码为每个 token 分配一个唯一的位置信息，而相对位置编码则关注 token 之间的相对位置关系。

在本节中，我们将实现一个简单的绝对位置编码方法，并将其集成到我们的词嵌入层中，以便在训练语言模型时能够同时获取 token


In [27]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [28]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


简单来说，上面的代码实现了一个词嵌入层，将 token id 转换为嵌入向量。接下来要定义一个位置编码层，将位置信息添加到词嵌入中，以便模型能够理解 token 之间的相对位置关系。

In [29]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


LLM所需要的输入是词嵌入和位置编码的和，因此我们需要将词嵌入和位置编码相加，得到最终的输入表示。

In [30]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## 📌 本章总结

### 核心收获

本章完成了从**原始文本**到**模型输入**的完整数据预处理流程，这是构建 LLM 的基础设施：

1. **分词策略演进**：从简单的正则分词 → 处理未知词的 `<UNK>` 标记 → 工业级的 BPE 子词编码，理解了不同分词方法的权衡
2. **数据采样机制**：通过滑动窗口将长文本切分为重叠的训练样本，实现了"预测下一个词"的自监督学习任务构造
3. **向量化表示**：Embedding 层将离散的 Token ID 映射为连续的向量空间，位置编码补充了序列顺序信息

### 关键技术点

| 组件 | 作用 | 实现方式 |
|------|------|---------|
| Tokenizer | 文本→Token IDs | tiktoken (BPE) |
| Dataset | 滑动窗口采样 | `GPTDatasetV1` |
| DataLoader | 批量数据加载 | PyTorch DataLoader |
| Embedding | ID→向量 | `nn.Embedding(vocab_size, dim)` |
| Position Encoding | 添加位置信息 | `nn.Embedding(max_length, dim)` |

### 数据流转全景
- 原始文本
- [BPE 分词] Token IDs: [15239, 995, 432, ...]
- [滑动窗口采样] Batch：inputs : [8 × 4] targets: [8 × 4]
- [Token Embedding] Tensor: [8 × 4 × 256]
- [+ Position Embedding]
- 最终输入表示: [8 × 4 × 256] → Transformer
### 下一步

有了准备好的数据输入管道，下一章将构建 **Transformer 架构的核心组件**（注意力机制、前馈网络等），正式进入 LLM 模型实现阶段。

